# Task: GNN training on synthetic SPV simulations: adjacency, cell state and property matrices

We will be developing a graph neural network (GNN)-based model capable of inferring mechanistic rules and uncovering the principles driving DPAC aggregation. To facilitate this, the GNN will initially be trained using synthetic Self-Propelled Voronoi (SPV) simulations, serving as placeholder data while the deep learning infrastructure is optimized. The GNN will be validated by its ability to, first, recover the physical mechanisms embedded in the SPV model, then subsequently applied to DPAC data to explore the impacts of initial thickness and cell density.

### GNN training

In [15]:
import os
import numpy as np
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F

# Use the standard PyTorch DataLoader:
from torch.utils.data import DataLoader

# We'll need Batch.from_data_list to merge multiple PyG Data objects
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.utils import from_networkx

from torch.optim import AdamW, lr_scheduler
from sklearn.model_selection import KFold

##########################################################################
# Custom collate function for standard DataLoader
##########################################################################

def pyg_collate_fn(batch_list):
    """
    Merges a list of PyG Data objects into a single `Batch`.
    Useful when using the standard torch.utils.data.DataLoader.
    """
    return Batch.from_data_list(batch_list)

##########################################################################
# 1. Parameter Parsing
##########################################################################

def parse_parameters(param_path):
    """
    Parse parameters from the given file.
    Ensures that W is converted to a NumPy array if it isn't already.
    """
    params = {}
    print(f"Parsing parameters from {param_path}")
    with open(param_path, "r") as f:
        exec(f.read(), {}, params)  # Execute the file content in a controlled namespace
    if not isinstance(params["W"], np.ndarray):
        params["W"] = np.array(params["W"])  # Convert W to NumPy array
    print(f"Successfully parsed parameters: {params}")
    return params

##########################################################################
# 2. Data Loading with Caching
##########################################################################

def load_matrices(directory, timepoint):
    """Load matrices with caching for faster subsequent access"""
    cache_key = (directory, timepoint)
    if not hasattr(load_matrices, 'cache'):
        load_matrices.cache = {}
    if cache_key not in load_matrices.cache:
        print(f"Loading matrices for timepoint {timepoint} from {directory}")
        graph_mat = np.load(os.path.join(directory, f"{timepoint}_graph_mat.npy"))
        properties_mat = np.load(os.path.join(directory, f"{timepoint}_properties_mat.npy"))
        state_mat = np.load(os.path.join(directory, f"{timepoint}_state_mat.npy"))
        load_matrices.cache[cache_key] = (graph_mat, properties_mat, state_mat)
    return load_matrices.cache[cache_key]

##########################################################################
# 3. GCA Initialization
##########################################################################

def initialize_gca(graph_mat, properties_mat, state_mat, params):
    """
    Create a graph (GCA) with proper state handling and parameter validation.
    - Validates non-negative area and perimeter.
    - Stores cell type distribution (state).
    - Adds area/perimeter and relevant model parameters (v0, Dr, kappa_A, etc.).
    - Creates edges with adhesive and repulsive properties from 'params'.
    """
    print("Initializing GCA graph...")
    g = nx.from_numpy_array(graph_mat, create_using=nx.Graph)
    
    # Validate physical constraints
    if np.any(properties_mat[:, 0] < 0) or np.any(properties_mat[:, 1] < 0):
        raise ValueError("Area and perimeter must be non-negative")

    # Add node properties with proper state handling
    print("Adding node properties...")
    for i, (area, perimeter) in enumerate(properties_mat):
        if area < 0 or perimeter < 0:
            raise ValueError(f"Invalid properties at node {i}: area={area}, perimeter={perimeter}")
        
        cell_type = np.argmax(state_mat[i])  # Argmax of the probability distribution
        g.nodes[i]["state"] = state_mat[i]   # Full state vector
        g.nodes[i].update({
            "area": max(area, 0),
            "perimeter": max(perimeter, 0),
            "motility": params["v0"][cell_type],  # from v0 array
            "persistence": params["Dr"],
            "kappa_A": params["kappa_A"],
            "kappa_P": params["kappa_P"],
            "A0": params["A0"][cell_type],
            "P0": params["P0"][cell_type],
        })

    # Add edge properties with validation
    print("Adding edge properties...")
    for u, v in g.edges():
        type_u = np.argmax(g.nodes[u]["state"])
        type_v = np.argmax(g.nodes[v]["state"])
        adhesion = params["W"][type_u][type_v]  # adhesion from W matrix
        g.edges[u, v].update({
            "adhesion": adhesion,
            "repulsion_radius": params["a"], 
            "repulsion_coefficient": params["k"],
        })

    # Store adjacency matrix for reference
    g.graph["adj_matrix"] = graph_mat
    print("GCA graph initialization complete.")
    return g

##########################################################################
# 4. Define the GraphPredictor Model
##########################################################################

class GraphPredictor(nn.Module):
    """
    A graph neural network to predict:
      1) The distribution of next cell types (state probabilities),
      2) Next area and perimeter,
      3) Changes in adjacency (focusing on clustering).
    
    node_dim:  Dimensionality of node features (x)
    edge_dim:  Dimensionality of edge features (edge_attr)
    hidden_dim: Size of hidden layers in GNN
    num_cell_types: Number of possible cell types (for state distribution)
    """
    def __init__(self, node_dim, edge_dim, hidden_dim, num_cell_types):
        super(GraphPredictor, self).__init__()
        
        # Encoders for node and edge features
        self.node_encoder = nn.Linear(node_dim, hidden_dim)
        self.edge_encoder = nn.Linear(edge_dim, hidden_dim)
        
        # Two GCNConv layers (you can experiment with more or different layers)
        self.conv1 = GCNConv(hidden_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        
        # MLP heads for each prediction
        self.state_decoder = nn.Linear(hidden_dim, num_cell_types)
        self.area_decoder  = nn.Linear(hidden_dim, 1)
        self.perim_decoder = nn.Linear(hidden_dim, 1)
        
        # For adjacency changes, we can do a pairwise MLP after node embeddings
        # Or (as a simpler approach) decode adjacency from final node embeddings 
        # in a "pseudo-edge" manner. We'll do a simpler approach: each edge gets 
        # a representation from node embeddings (u||v). Then we use a linear layer.
        self.adj_decoder = nn.Linear(2 * hidden_dim, 1)
    
    def forward(self, x, edge_index, edge_attr, batch=None):
        """
        x:          node features [num_nodes, node_dim]
        edge_index: COO format [2, num_edges]
        edge_attr:  edge features [num_edges, edge_dim]
        batch:      (Optional) batch indices if multiple graphs in a batch

        Returns: state_pred, area_pred, perim_pred, adj_pred
        - state_pred:     [num_nodes, num_cell_types]
        - area_pred:      [num_nodes, 1]
        - perim_pred:     [num_nodes, 1]
        - adj_pred:       [num_edges, 1] (predicted adjacency or existence probability)
        """
        # Encode node and edge features
        x = self.node_encoder(x)
        e = self.edge_encoder(edge_attr)
        
        # First GCN layer
        x = self.conv1(x, edge_index, e)
        x = F.relu(x)
        # Second GCN layer
        x = self.conv2(x, edge_index, e)
        x = F.relu(x)
        
        # Decode per-node predictions
        state_pred = F.log_softmax(self.state_decoder(x), dim=-1)  # for KLDiv
        area_pred  = self.area_decoder(x)
        perim_pred = self.perim_decoder(x)
        
        # Build adjacency predictions for edges
        row, col = edge_index
        x_u = x[row]
        x_v = x[col]
        edge_repr = torch.cat([x_u, x_v], dim=1)
        adj_pred = self.adj_decoder(edge_repr)
        
        return state_pred, area_pred, perim_pred, adj_pred

##########################################################################
# 5. Training & Validation
##########################################################################

def train_epoch(model, data_list, optimizer, device, num_cell_types):
    """
    One training epoch over a list of PyG data objects. 
    Focuses on state, area, perimeter, and adjacency loss.
    
    Uses the standard torch DataLoader with a custom collate_fn to merge 
    multiple PyG Data objects into a single Batch object.
    """
    model.train()
    total_loss = 0
    
    # Standard torch DataLoader, custom collate_fn
    loader = DataLoader(data_list, batch_size=1, shuffle=True, collate_fn=pyg_collate_fn)
    
    # Define separate loss functions
    criterion_state = nn.KLDivLoss(reduction='batchmean')  # cell-type distribution
    criterion_prop  = nn.SmoothL1Loss()                    # area, perimeter
    criterion_adj   = nn.BCEWithLogitsLoss()               # adjacency existence
    
    for batch_idx, batch in enumerate(loader):
        optimizer.zero_grad()
        
        # Move data to device
        x = batch.x.to(device)
        edge_index = batch.edge_index.to(device)
        edge_attr = batch.edge_attr.to(device)
        
        # Collect training targets
        next_state = batch.next_state.to(device)
        next_area  = batch.next_area.unsqueeze(-1).to(device)    # shape [num_nodes, 1]
        next_perim = batch.next_perim.unsqueeze(-1).to(device)   # shape [num_nodes, 1]
        next_adj   = batch.next_adj.to(device)                   # shape [num_edges]
        
        # Forward pass
        state_pred, area_pred, perim_pred, adj_pred = model(x, edge_index, edge_attr)
        
        # Compute losses
        loss_state = criterion_state(state_pred, next_state)
        loss_area  = criterion_prop(area_pred, next_area)
        loss_perim = criterion_prop(perim_pred, next_perim)
        loss_adj   = criterion_adj(adj_pred.squeeze(), next_adj.float())
        
        # Combine
        total_batch_loss = loss_state + loss_area + loss_perim + loss_adj
        
        # L2 regularization
        l2_reg = 1e-4 * sum(p.pow(2.0).sum() for p in model.parameters())
        total_batch_loss += l2_reg
        
        total_batch_loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
        optimizer.step()
        
        total_loss += total_batch_loss.item()
        
        if (batch_idx + 1) % 10 == 0:
            print(f"  [Train] Batch {batch_idx + 1}/{len(loader)} - Loss: {total_batch_loss.item():.4f}")
    
    avg_loss = total_loss / len(loader)
    print(f"  [Train] Epoch complete. Average loss: {avg_loss:.4f}")
    return avg_loss

def validate(model, data_list, device, num_cell_types):
    """
    Validation loop using the same set of losses.
    Also uses the standard DataLoader with custom collate_fn for PyG Batches.
    """
    model.eval()
    total_loss = 0
    
    loader = DataLoader(data_list, batch_size=1, shuffle=False, collate_fn=pyg_collate_fn)
    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            # Move data to device
            x = batch.x.to(device)
            edge_index = batch.edge_index.to(device)
            edge_attr = batch.edge_attr.to(device)
            
            next_state = batch.next_state.to(device)
            next_area  = batch.next_area.unsqueeze(-1).to(device)
            next_perim = batch.next_perim.unsqueeze(-1).to(device)
            next_adj   = batch.next_adj.to(device)
            
            # Forward
            state_pred, area_pred, perim_pred, adj_pred = model(x, edge_index, edge_attr)
            
            # Losses
            loss_state = F.kl_div(state_pred, next_state, reduction='batchmean')
            loss_area  = F.l1_loss(area_pred, next_area)
            loss_perim = F.l1_loss(perim_pred, next_perim)
            loss_adj   = F.binary_cross_entropy_with_logits(adj_pred.squeeze(), next_adj.float())
            
            total_loss += (loss_state + loss_area + loss_perim + loss_adj).item()
            
            if (batch_idx + 1) % 10 == 0:
                print(f"  [Val] Batch {batch_idx + 1}/{len(loader)} - Cumulative Loss: {total_loss:.4f}")
    
    avg_loss = total_loss / len(loader)
    print(f"  [Val] Validation complete. Average loss: {avg_loss:.4f}")
    return avg_loss

##########################################################################
# 6. K-Fold Training
##########################################################################

def k_fold_training(
    data_dirs, param_files, timepoints, timepoint_interval,
    node_dim, edge_dim, hidden_dim, num_cell_types,
    epochs=100, lr=1e-3, k_folds=5
):
    """
    data_dirs, param_files: lists of directories and param files for each dataset
    timepoints: total number of timepoints you have
    timepoint_interval: spacing between timepoints for constructing (current -> next) pairs
    node_dim, edge_dim, hidden_dim: dimensionalities for the GNN
    num_cell_types: how many cell types are possible
    epochs, lr, k_folds: training hyperparameters
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # Preprocess all data
    print("Preprocessing data...")
    all_data = []
    for dir_idx, (data_dir, param_file) in enumerate(zip(data_dirs, param_files)):
        print(f"Processing directory {data_dir} with parameter file {param_file}...")
        params = parse_parameters(param_file)
        
        # Build training pairs for each consecutive interval
        for t in range(0, timepoints - timepoint_interval, timepoint_interval):
            print(f"Loading timepoint {t} and {t + timepoint_interval}...")
            current = load_matrices(data_dir, t)
            next_t = load_matrices(data_dir, t + timepoint_interval)
            
            # Initialize GCAs
            print(f"Initializing GCA for timepoint {t}...")
            g_current = initialize_gca(*current, params)
            g_next = initialize_gca(*next_t, params)
            
            # Convert to PyG data
            data = from_networkx(g_current)
            
            # Next-state info from g_next
            data.next_state = torch.tensor(
                np.array([g_next.nodes[i]["state"] for i in g_current.nodes]),
                dtype=torch.float
            )
            data.next_area = torch.tensor(
                [g_next.nodes[i]["area"] for i in g_current.nodes],
                dtype=torch.float
            )
            data.next_perim = torch.tensor(
                [g_next.nodes[i]["perimeter"] for i in g_current.nodes],
                dtype=torch.float
            )
            # Flatten adjacency of g_next
            data.next_adj = torch.tensor(
                nx.to_numpy_array(g_next)[data.edge_index[0], data.edge_index[1]],
                dtype=torch.float
            )
            
            # Adjust or check x, edge_attr dimension
            # Here you might ensure x has dimension = node_dim, edge_attr has dimension = edge_dim
            
            # For demonstration, let's create node features from area/perimeter + state
            # So node_dim = (2 + num_cell_types) if we want area, perimeter, and state
            node_feat = []
            for i in g_current.nodes:
                node_feat.append(np.concatenate([
                    [g_current.nodes[i]["area"], g_current.nodes[i]["perimeter"]],
                    g_current.nodes[i]["state"]
                ]))
            data.x = torch.tensor(node_feat, dtype=torch.float)
            
            # Create edge features from e.g. adhesion, repulsion_radius, etc.
            edge_feat = []
            for (u, v) in zip(data.edge_index[0], data.edge_index[1]):
                adh = g_current.edges[int(u), int(v)]["adhesion"]
                rep_r = g_current.edges[int(u), int(v)]["repulsion_radius"]
                rep_k = g_current.edges[int(u), int(v)]["repulsion_coefficient"]
                edge_feat.append([adh, rep_r, rep_k])
            data.edge_attr = torch.tensor(edge_feat, dtype=torch.float)
            
            all_data.append(data)

    for i, d in enumerate(all_data):
        print(i, type(d))
    # Possibly also do: 
    #   if not isinstance(d, Data): 
    #       print("This is not a torch_geometric.data.Data!", d)


    # K-fold training
    kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)
    best_model = None
    best_loss = float('inf')
    
    data_array = np.array(all_data, dtype=object)
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(data_array)):
        print(f"\n--- Starting Fold {fold+1}/{k_folds} ---")
        train_data = data_array[train_idx].tolist()
        val_data = data_array[val_idx].tolist()
        
        # Right after splitting into train_data and val_data:
        print("Checking train_data objects:")
        for i, d in enumerate(train_data):
            print(i, type(d))

        print("Checking val_data objects:")
        for i, d in enumerate(val_data):
            print(i, type(d))


        print(f"  Training data size: {len(train_data)}")
        print(f"  Validation data size: {len(val_data)}")
        
        # Initialize model, optimizer, scheduler
        model = GraphPredictor(node_dim, edge_dim, hidden_dim, num_cell_types).to(device)
        optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)
        
        best_val_loss = float('inf')
        patience_counter = 0
        for epoch in range(epochs):
            print(f"\nEpoch {epoch+1}/{epochs}")
            train_loss = train_epoch(model, train_data, optimizer, device, num_cell_types)
            val_loss = validate(model, val_data, device, num_cell_types)
            scheduler.step(val_loss)
            
            print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
            
            # Early stopping
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                torch.save(model.state_dict(), f"best_fold{fold}.pth")
                print(f"    --> New best model saved for fold {fold+1} with validation loss: {best_val_loss:.4f}")
            else:
                patience_counter += 1
                if patience_counter >= 10:
                    print("    --> Early stopping triggered.")
                    break
        
        # Update best overall model
        if best_val_loss < best_loss:
            best_loss = best_val_loss
            best_model = model
    
    print(f"\nTraining complete. Best validation loss across folds: {best_loss:.4f}")
    
    if best_model is not None:
        torch.save(best_model.state_dict(), "best_model.pth")
        print("Best model saved to 'best_model.pth'")
    else:
        print("No best model found (should not happen unless no training occurred).")
    
    return best_model

##########################################################################
# 7. Main Execution
##########################################################################

if __name__ == "__main__":
    # Example usage:
    
    # Suppose you have 10 data directories, each with an associated param file:
    data_dirs = [
        f"/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I_matrix/{i}_matrix_output"
        for i in range(1, 11)
    ]
    param_files = [
        f"/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/{i}.py"
        for i in range(1, 11)
    ]
    # Hypothetical total timepoints and interval
    timepoints = 2000
    timepoint_interval = 500
    
    # Suppose your node features have dimension 2 + num_cell_types 
    # (area, perimeter + state distribution)
    num_cell_types = 3
    node_dim = 2 + num_cell_types  # area + perimeter + distribution
    # For edge features, e.g. (adhesion, repulsion_radius, repulsion_coefficient)
    edge_dim = 3  
    hidden_dim = 64
    
    # Run K-Fold Training
    best_model = k_fold_training(
        data_dirs, param_files,
        timepoints, timepoint_interval,
        node_dim, edge_dim, hidden_dim, num_cell_types,
        epochs=30, lr=1e-3, k_folds=3
    )

    print("Done.")


Using device: cpu
Preprocessing data...
Processing directory /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I_matrix/1_matrix_output with parameter file /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/1.py...
Parsing parameters from /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/1.py
Successfully parsed parameters: {'domain_size': [60, 14], 'init_noise': 0.005, 'rng_seed': 1, 'dt': 0.25, 'tMax': 501, 'stripe_thickness': 4, 'stripe_density': 0.5, 'v0': [0.1, 1.3], 'W': array([[0.  , 0.08],
       [0.08, 0.  ]]), 'A0': [0.9, 0.9], 'P0': [3.812, 3.812], 'Dr': 50, 'kappa_A': 0.4, 'kappa_P': 0.07, 'a': 0.25, 'k': 2.5}
Loading timepoint 0 and 500...
Loading matrices for timepoint 0 from /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I_matrix/1_matrix_output
Loading matrices for timepoint 500 from /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output

AttributeError: 'listBatch' object has no attribute 'stores_as'

In [2]:
import torch
from torch.utils.data import DataLoader
from torch_geometric.data import Data, Batch

def pyg_collate_fn(batch_list):
    return Batch.from_data_list(batch_list)

# Make a single small Data object
data = Data(x=torch.rand(3,2), edge_index=torch.tensor([[0,1],[1,2]]))

loader = DataLoader([data], batch_size=1, collate_fn=pyg_collate_fn)
for i, b in enumerate(loader):
    print("Type of batch:", type(b))
    print("Keys:", b.keys)


Type of batch: <class 'abc.DataBatch'>
Keys: <bound method BaseData.keys of DataBatch(x=[3, 2], edge_index=[2, 2], batch=[3], ptr=[2])>


In [ ]:
if __name__ == "__main__":
    # Directories and parameter files
    data_dirs = [
        f"/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I_matrix/{i}_matrix_output"
        for i in range(1, 11)
    ]
    param_files = [
        f"/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/{i}.py"
        for i in range(1, 11)
    ]

    # Run the main training process
    trained_model = main(data_dirs, param_files)

    # Save the final trained model
    torch.save(trained_model.state_dict(), "final_trained_model.pth")
    print("Final model saved to 'final_trained_model.pth'")

In [ ]:
import os

# Check if the files exist
directory = "/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I_matrix/1_matrix_output"
timepoint = 0  # Replace with the correct timepoint
required_files = [
    f"{timepoint}_graph_mat.npy",
    f"{timepoint}_properties_mat.npy",
    f"{timepoint}_state_mat.npy"
]

for file in required_files:
    file_path = os.path.join(directory, file)
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
    else:
        print(f"File exists: {file_path}")